In [ ]:
!pip install ultralytics
#!pip install matplotlib
#pip install pandas


In [ ]:
from ultralytics import YOLO
# Load YOLO11 segmentation model
model = YOLO("yolov8s-seg.pt")  # other options: yolo11m-seg.pt, yolo11l-seg.pt, yolo11x-seg.pt

In [ ]:
model.info()

In [ ]:
from ultralytics import YOLO
from ultralytics.nn.modules import CBAM
import torch.nn as nn


model = YOLO("yolov8s-seg-custom.yaml")
model.info()


#model.train(data="./merged_dataset/data.yaml", epochs=100, imgsz=640, batch=24)

In [ ]:
from ultralytics import YOLO

# Load the model
model = YOLO('yolov8s-seg.pt')
pytorch_model = model.model

# YOLOv8-seg has a specific segmentation head
print("YOLOv8s-seg Model Components:")
print("=" * 50)

# Print the entire model structure with focus on segmentation head
print("\nModel Structure:")
print(pytorch_model)

# Specifically look at the model's head (where segmentation happens)
if hasattr(pytorch_model, 'model'):
    # For YOLOv8, the last part is typically the head
    print("\nSegmentation Head Layers:")
    for i, layer in enumerate(pytorch_model.model[-1:]):
        print(f"Head Layer {i}: {layer}")
        
        # If it's the Detect or Segment module, show its structure
        if hasattr(layer, 'cv2') and hasattr(layer, 'cv3'):
            print(f"  - Detection convolutions: {layer.cv2}")
            print(f"  - Segmentation convolutions: {layer.cv3}")

In [ ]:
model.load('./runs/segment/train6/weights/best.pt')

In [ ]:
import torch
from ultralytics import YOLO
import numpy as np

def debug_model():
    # Create model
    model = YOLO("yolov8s-seg-custom.yaml")
    
    # Test 1: Check input/output shapes
    print("Test 1: Forward pass with dummy data")
    dummy = torch.randn(2, 3, 640, 640)
    
    try:
        with torch.no_grad():
            outputs = model.model(dummy)
            print(f"Success! Output types: {type(outputs)}")
            
            if isinstance(outputs, tuple):
                boxes, masks = outputs
                print(f"Boxes shape: {boxes.shape}")
                print(f"Masks shape: {masks.shape}")
    except Exception as e:
        print(f"Forward pass error: {e}")
    
    # Test 2: Check if it's a training-time issue
    print("\nTest 2: Simulating training loss calculation")
    
    # Create fake predictions and targets
    pred_masks = torch.randn(2, 32, 160, 160)  # Example mask prediction
    target_masks = torch.randn(2, 1, 640, 640)  # Example ground truth
    
    # Try to calculate IoU (this is where your error occurs)
    print(f"Pred masks shape: {pred_masks.shape}")
    print(f"Target masks shape: {target_masks.shape}")
    
    # Test 3: Check BatchNorm in frozen backbone
    print("\nTest 3: Checking BatchNorm layers")
    for name, module in model.model.named_modules():
        if isinstance(module, torch.nn.BatchNorm2d):
            print(f"BatchNorm: {name}")
            print(f"  Training: {module.training}")
            print(f"  Requires grad: {module.weight.requires_grad}")

debug_model()

In [ ]:
import yaml

# Load the two YAML files
with open("tacodatasetv2/data.yaml") as f:
    data0 = yaml.safe_load(f)
with open("trashdatasetbig/data.yaml") as f:
    data1 = yaml.safe_load(f)
with open("tacodatasetv1/data.yaml") as f:
    data2 = yaml.safe_load(f)
with open("trashcanholedataset/data.yaml") as f:
    data3 = yaml.safe_load(f)

# Get the class names
classes0 = set(data0["names"])
classes1 = set(data1["names"])
classes2 = set(data2["names"])
classes3 = set(data3["names"])


# Union of the two sets
merged_classes = classes0 | classes1 | classes2 | classes3  # or classes1.union(classes2)

# Print the merged set
print(merged_classes)
#for cls in classes0 ^ classes2:
#    print(cls)

# papper straw plastic straw straw

In [ ]:
import os
import shutil

Paper = ["Paper", "Normal paper", "Paper cup", "Paper straw", "Paper bag", "Carded blister pack", "Magazine paper", "Wrapping paper", "Tissues"]
Carton = ["Corrugated carton", "Meal carton", "Pizza box", "Carton", "Other carton", "Egg carton", "Drink carton"]
Plastic =  ["Blister pack","Plastic container", "Plastic lid", "Plastic straw", "Plastic bottle cap", "Other plastic cup", "Other plastic bottle", "Plasticbag-wrapper", "Single-use carrier bag", "Other plastic wrapper", "Clear plastic bottle", "Polypropylene bag", "Tupperware", "Squeezable tube", "Disposable plastic cup", "Plastic film", "Plastic utensils", "Foam food container", "Foam cup", "Styrofoam piece", "Disposable food container", "Plastic glooves", "Other plastic", "Bottle cap", "Other plastic container", "Six pack rings"]
Glass = ["Glass bottle", "Glass jar", "Glass cup", "Broken glass"]
Metal = ["Food Can", "Drink can", "Aerosol", "Scrap metal", "Metal lid", "Metal bottle cap", "Pop tab", "Aluminium foil", "Aluminium blister pack", "Can"]
Organic =  ["Food waste", "Spread tub"]
Cigarette = ["Cigarette"]
Trashcan = ["trashcan-holes"]
Small_Uncategorized_items = ["Straw", "Rope - strings", "Bottle","Cup", "Lid", "Crisp packet", "Battery", "Toilet tube"]
Unlabeled_litter = ["Unlabeled litter","Shoe"]
Garbage_bag = ["Garbage bag"]

categories = [
    Paper, Carton, Plastic, Glass, Metal,
    Organic, Cigarette, Trashcan,
    Small_Uncategorized_items, Unlabeled_litter, Garbage_bag
]

categories_names = ["Paper", "Carton", "Plastic", "Glass", "Metal",
    "Organic", "Cigarette", "Trashcan",
    "Small_Uncategorized_items", "Unlabeled_litter", "Garbage_bag"]

dataset_paths = [
    "./tacodatasetv2",
    "./trashdatasetbig",
    "./tacodatasetv1",
    "./trashcanholedataset",
]

out_root = "./merged_dataset"

# Output folders (YOLOv11 style)
splits = ["train", "valid"]
for split in splits:
    os.makedirs(os.path.join(out_root, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(out_root, split, "labels"), exist_ok=True)

class_map = {}
for new_id, group in enumerate(categories):
    for name in group:
        class_map[name] = new_id

print(class_map)

# ===========================
# MERGE + REMAP
# ===========================
img_counter = {"train": 0, "valid": 0}

for dataset in dataset_paths:
    # Load original class names
    with open(os.path.join(dataset, "data.yaml")) as f:
        import yaml
        old_names = yaml.safe_load(f)["names"]
        old_names = list(old_names)

    for split in splits:
        img_src = os.path.join(dataset, split, "images")
        lbl_src = os.path.join(dataset, split, "labels")

        for img in os.listdir(img_src):
            if not img.lower().endswith((".jpg",".png",".jpeg")):
                continue

            new_name = f"{img_counter[split]:07d}.jpg"
            img_counter[split] += 1

            # Copy image
            shutil.copy2(
                os.path.join(img_src, img),
                os.path.join(out_root, split, "images", new_name)
            )

            # Remap label
            src_lbl = os.path.join(lbl_src, img.rsplit(".",1)[0] + ".txt")
            dst_lbl = os.path.join(out_root, split, "labels", new_name.replace(".jpg",".txt"))

            if not os.path.exists(src_lbl):
                continue

            new_lines = []
            with open(src_lbl) as f:
                for line in f:
                    parts = line.split()
                    old_id = int(parts[0])
                    old_name = old_names[old_id]
                    if old_name in class_map:
                        new_id = class_map[old_name]
                        new_lines.append(" ".join([str(new_id)] + parts[1:]))

            with open(dst_lbl, "w") as f:
                f.write("\n".join(new_lines))

# ===========================
# CREATE data.yaml
# ===========================
with open(os.path.join(out_root, "data.yaml"), "w") as f:
    f.write(
        f"""train: train/images
val: valid/images

nc: {len(categories)}
names:
"""
    )
    for i, group in enumerate(categories):
        f.write(f"  {i}: {categories_names[i]}\n")

print("✅ YOLOv11 dataset created:", out_root)

In [ ]:
import os
from collections import Counter
import yaml

# Load class names
with open("./merged_dataset/data.yaml") as f:
    data = yaml.safe_load(f)

names = {int(k): v for k, v in data["names"].items()}

# Paths to label folders
label_dirs = [
    "./merged_dataset/train/labels",
    #"./merged_dataset/valid/labels"
]

counter = Counter()

for lbl_dir in label_dirs:
    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith(".txt"):
            continue
        path = os.path.join(lbl_dir, lbl_file)
        with open(path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 0:
                    continue
                cls_id = int(parts[0])
                cls_name = names.get(cls_id, f"unknown_{cls_id}")
                counter[cls_name] += 1

# Print frequencies
for cls_name, count in sorted(counter.items()):
    print(f"{cls_name}: {count}")


In [ ]:
import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt
from ultralytics.data.dataset import YOLODataset

# Load YAML
with open("./merged_dataset/data.yaml") as f:
    data = yaml.safe_load(f)

dataset = YOLODataset(
    img_path="./merged_dataset/merged_dataset/train/images",
    data=data,
    task="segment"
)

names = dataset.data["names"]
TARGET_CLASS = "trashcan-holes"

# Collect only images containing target class
items = []
for img_path, label in zip(dataset.im_files, dataset.labels):
    if any(names[int(c)] == TARGET_CLASS for c in label.get("cls", [])):
        items.append((img_path, label))

print(f"Found {len(items)} images with {TARGET_CLASS}")

idx = 0
fig, ax = plt.subplots(figsize=(7, 7))

def draw():
    ax.clear()
    img_path, label = items[idx]

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    ax.imshow(img)

    for cls_id, segment in zip(label["cls"], label["segments"]):
        if names[int(cls_id)] != TARGET_CLASS:
            continue

        pts = np.array(segment).reshape(-1, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        ax.plot(*pts.T, color="red")
        ax.fill(*pts.T, color="red", alpha=0.3)

    ax.set_title(f"{idx+1}/{len(items)} — {img_path.split('/')[-1]}")
    ax.axis("off")
    fig.canvas.draw_idle()

def on_key(event):
    global idx
    if event.key in ["right", "enter", " "]:
        idx = min(idx + 1, len(items) - 1)
    elif event.key == "left":
        idx = max(idx - 1, 0)
    elif event.key in ["escape", "q"]:
        plt.close(fig)
        return
    draw()

fig.canvas.mpl_connect("key_press_event", on_key)
draw()
plt.show()


In [ ]:
import torch
print(torch.version.cuda) # CUDA version PyTorch was built with
print(torch.cuda.is_available()) # True if a GPU is detected
print(torch.cuda.device_count()) # Number of GPUs

In [ ]:
# Train on COCO-format dataset
model.train(
    data="./merged_dataset/merged_dataset/data.yaml",
    epochs=150,
    imgsz=640,
    #save_period=10,
    batch=24,
    project="./runs_custom",
    #resume=True,
)

In [ ]:
import os

# Path to your label folder
label_dir = "./trashcan_hole_dataset/test/labels"

# List to store bad files
bad_files = []

for f in os.listdir(label_dir):
    if not f.endswith(".txt"):
        continue
    path = os.path.join(label_dir, f)
    with open(path, "r") as file:
        lines = file.readlines()
        for line in lines:
            parts = line.strip().split()[1:]  # remove class_id
            # Check for 4 numbers (detect bbox) or odd number (malformed polygon)
            if len(parts) == 4 or len(parts) % 2 != 0:
                bad_files.append(path)
                break  # no need to check the rest of the file

# Delete bad files
for f in bad_files:
    os.remove(f)
    print(f"Deleted bad label file: {f}")

print(f"Total deleted: {len(bad_files)}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a simple test image (RGB)
img = np.zeros((300, 300, 3), dtype=np.uint8)

# Draw something visible
img[:] = [255, 0, 0]        # Red background
img[50:250, 50:250] = [0, 255, 0]  # Green square inside

# Plot it
plt.imshow(img)
plt.title("Test Image")
plt.axis("off")
plt.show()
plt.imsave("test_output.png", img)

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Load model
model = YOLO("./runs/segment/train6/weights/best.pt")

# Run prediction on ONE image
results = model.predict(
    source="./merged_dataset/merged_dataset/valid/images/0000182.jpg",
    conf=0.2,
    save=False,
    show=False
)

print(results)

# Get plotted image (with predictions drawn)
img_with_preds = results[0].plot()  # returns numpy array (BGR)

# Show with matplotlib
plt.figure(figsize=(7, 7))
plt.imshow(img_with_preds[..., ::-1])  # BGR -> RGB
plt.axis("off")
plt.show()
plt.imsave("test_output2.png", img_with_preds[..., ::-1])

In [ ]:
from ultralytics import YOLO
from collections import defaultdict

model = YOLO("./runs/segment/train44/weights/best.pt")

results = model.predict("testedData/video3.mp4", conf=0.2,save = False)

class_counts = defaultdict(int)
confidence_sum = defaultdict(float)
total_detections = 0

for r in results:
    if r.boxes is None:
        continue

    for cls, conf in zip(r.boxes.cls, r.boxes.conf):
        cls = int(cls)
        class_counts[cls] += 1
        confidence_sum[cls] += float(conf)
        total_detections += 1

# Convert class IDs → names
names = model.names

print("📊 Summary:")
for cls, count in class_counts.items():
    avg_conf = confidence_sum[cls] / count
    print(f"{names[cls]}: {count} detections, avg conf = {avg_conf:.2f}")

print(f"\nTotal detections: {total_detections}")

In [ ]:
from ultralytics import YOLO

model = YOLO("./runs/segment/train44/weights/best.pt")
model.export(
    format="onnx",
    imgsz=640,
    opset=21,
    device='cpu',
)

In [ ]:
import onnx

# Load your ONNX model
onnx_model = onnx.load("./runs/segment/train6/weights/best.onnx")

# Print all outputs
for output in onnx_model.graph.output:
    print(f"Output name: {output.name}, type: {output.type}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Paths to your CSV files
csv_files = [
    "runs/segment/train6/results.csv",
    "runs/segment/train44/results.csv",
    "runs/segment/train40/results.csv"
]

# Create the plot
plt.figure(figsize=(10, 5))

# Loop through CSV files and plot the same metric from each
df = pd.read_csv(csv_files[0])
plt.plot(df['epoch'], df['metrics/mAP50(B)'], marker='o', label=f'mAP50 unmodified model')
df = pd.read_csv(csv_files[1])
plt.plot(df['epoch'], df['metrics/mAP50(B)'], marker='o', label=f'mAP50 our model')
df = pd.read_csv(csv_files[2])
plt.plot(df['epoch'], df['metrics/mAP50(B)'], marker='o', label=f'mAP50 Yolov8m-seg')

plt.xlabel('Epoch')
plt.ylabel('mAP50 (B)')
plt.title('YOLO mAP50 (B) Comparison Across Training Runs')
plt.grid(True)
plt.legend()

# Save the combined plot
plt.savefig("mAP50_B_comparison.png", dpi=300)
plt.close()

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

class YOLOv8SegAnnotator:
    def __init__(self, model_path='./Data/yolov8s-seg.pt'):
        """Load a YOLOv8 segmentation model"""
        self.model = YOLO(model_path)
        self.latestResults = None
        self.latestImage = None

    def update(self, image_path):
        """Run inference on an image from file path"""
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Image not found: {image_path}")
        self.latestImage = image.copy()
        # Run prediction and take the first result
        self.latestResults = self.model.predict(image, verbose=False)[0]

    def GetDataRaw(self):
        """Return raw predictions as a list of dicts with bbox, score, label, mask"""
        if self.latestResults is None:
            return []

        predictions = []
        boxes = self.latestResults.boxes.xyxy.cpu().numpy()
        scores = self.latestResults.boxes.conf.cpu().numpy()
        labels = self.latestResults.boxes.cls.cpu().numpy()
        masks = self.latestResults.masks.data.cpu().numpy() if hasattr(self.latestResults, "masks") else None

        for i in range(len(boxes)):
            pred = {
                "bbox": boxes[i],
                "score": float(scores[i]),
                "label": int(labels[i]),
                "mask": {"data": masks[i]} if masks is not None else None
            }
            predictions.append(pred)
        return predictions

    def draw_object_annotations(self, draw_mask_alpha=0.4):
        """Return annotated image with bounding boxes, masks, and PCA orientation lines (shorter lines)"""
        if self.latestResults is None or self.latestImage is None:
            return None
    
        image = self.latestImage.copy()
    
        for det in self.GetDataRaw():
            bbox = det['bbox']
            label = det.get('label', 'Unknown')
            score = det.get('score', 0.0)
            mask_data = det.get('mask', {}).get('data', None)
    
            x_min, y_min, x_max, y_max = [int(v) for v in bbox]
    
            # --- Draw bounding box + label ---
            cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
            cv2.putText(image, f"{label}: {score:.2f}", (x_min, max(y_min - 5, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
            # --- Overlay full-image mask ---
            if mask_data is not None:
                mask = (mask_data > 0).astype(np.uint8)
                mask_color = np.zeros_like(image, dtype=np.uint8)
                mask_color[:, :, 2] = mask * 255  # Red mask
    
                mask_bool = mask.astype(bool)
                image[mask_bool] = (image[mask_bool] * (1 - draw_mask_alpha) +
                                     mask_color[mask_bool] * draw_mask_alpha).astype(np.uint8)
    
                # --- PCA orientation line (shorter) ---
                ys, xs = np.where(mask > 0)
                if len(xs) > 0:
                    coords = np.column_stack((xs, ys)).astype(np.float32)
                    mean, eigenvectors = cv2.PCACompute(coords, mean=None)[:2]
                    center = tuple(mean[0].astype(int))
                    vec = eigenvectors[0]
    
                    # Shorter line: half of mask diagonal
                    mask_h, mask_w = mask.shape
                    length = 0.5 * np.sqrt(mask_h**2 + mask_w**2)
    
                    pt1 = (int(center[0] - vec[0] * length/10), int(center[1] - vec[1] * length/10))
                    pt2 = (int(center[0] + vec[0] * length/10), int(center[1] + vec[1] * length/10))
                    cv2.line(image, pt1, pt2, (255, 0, 0), 2)
                    cv2.circle(image, center, 3, (0, 255, 255), -1)
    
        return image


# --- Example usage ---
if __name__ == "__main__":
    annotator = YOLOv8SegAnnotator(model_path='runs/segment/train6/weights/best.pt')
    annotator.update('./merged_dataset/merged_dataset/valid/images/0000182.jpg')
    annotated_image = annotator.draw_object_annotations()
    cv2.imwrite('annotated.png', annotated_image)

In [ ]:
import cv2
import numpy as np
import random

# ----------------------------
# Load YOLO polygon annotations
# ----------------------------
def load_yolo_polygons(txt_path, img_w, img_h):
    polygons = []

    with open(txt_path, "r") as f:
        for line in f.readlines():
            parts = list(map(float, line.strip().split()))
            cls = int(parts[0])
            coords = parts[1:]

            pts = []
            for i in range(0, len(coords), 2):
                x = int(coords[i] * img_w)
                y = int(coords[i+1] * img_h)
                pts.append([x, y])

            polygons.append((cls, np.array(pts, dtype=np.int32)))

    return polygons

# ----------------------------
# Polygon → mask
# ----------------------------
def polygons_to_mask(polygons, shape):
    mask = np.zeros(shape[:2], dtype=np.uint8)

    for _, pts in polygons:
        cv2.fillPoly(mask, [pts], 1)

    return mask

# ----------------------------
# Extract boundary
# ----------------------------
def extract_boundary(mask, thickness=4):
    kernel = np.ones((3, 3), np.uint8)
    eroded = cv2.erode(mask, kernel, iterations=thickness)
    return mask - eroded

# ----------------------------
# Get random boundary from image2
# ----------------------------
def get_random_boundary(image, polygons):
    cls, pts = random.choice(polygons)

    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 1)

    boundary = extract_boundary(mask)
    boundary_pixels = image * boundary[:, :, None]

    return boundary, boundary_pixels

# ----------------------------
# Check overlap
# ----------------------------
def overlaps(a, b):
    return np.any((a + b) > 1)

# ----------------------------
# Main FakeMix function
# ----------------------------
def fakemix_from_paths(img1_path, txt1_path, img2_path, txt2_path, max_tries=30):
    # Load images
    img1 = cv2.imread(img1_path)
    img2 = cv2.imread(img2_path)

    h, w = img1.shape[:2]

    # Load annotations
    polys1 = load_yolo_polygons(txt1_path, w, h)
    polys2 = load_yolo_polygons(txt2_path, w, h)

    # Mask of real objects in image1
    mask1 = polygons_to_mask(polys1, img1.shape)

    # Get boundary from image2
    boundary, boundary_pixels = get_random_boundary(img2, polys2)

    for _ in range(max_tries):
        dx = random.randint(-w // 4, w // 4)
        dy = random.randint(-h // 4, h // 4)

        M = np.float32([[1, 0, dx], [0, 1, dy]])

        trans_boundary = cv2.warpAffine(boundary, M, (w, h))
        trans_pixels = cv2.warpAffine(boundary_pixels, M, (w, h))

        # Avoid overlapping real objects
        if overlaps(trans_boundary, mask1):
            continue

        # Paste
        inv = 1 - trans_boundary
        output = (img1 * inv[:, :, None]) + trans_pixels

        print("editted")

        return output.astype(np.uint8)

    print("default")
    return img1  # fallback

out = fakemix_from_paths(
    "./merged_dataset/merged_dataset/train/images/0000050.jpg", "./merged_dataset/merged_dataset/train/labels/0000050.txt",
    "./merged_dataset/merged_dataset/train/images/0000067.jpg", "./merged_dataset/merged_dataset/train/labels/0000067.txt"
)

cv2.imwrite("output.jpg", out)

In [ ]:
import os
import cv2
import yaml
import random
import numpy as np
from glob import glob

GLASS_CLASS_ID = 3  # change if needed
SKIP_CLASSES = [2,6,8,9]


# ----------------------------
# Resolve paths from data.yaml
# ----------------------------
def resolve_paths(data_yaml_path):
    with open(data_yaml_path, "r") as f:
        data = yaml.safe_load(f)

    yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))
    base_path = data.get("path", "")

    if base_path:
        base_path = os.path.join(yaml_dir, base_path)
    else:
        base_path = yaml_dir

    train_path = os.path.normpath(os.path.join(base_path, data["train"]))
    label_path = train_path.replace("images", "labels")

    return train_path, label_path


# ----------------------------
# Load YOLO polygons
# ----------------------------
def load_yolo_polygons(txt_path, img_w, img_h):
    polygons = []

    if not os.path.exists(txt_path):
        return polygons

    with open(txt_path, "r") as f:
        for line in f.readlines():
            parts = list(map(float, line.strip().split()))
            cls = int(parts[0])
            coords = parts[1:]

            pts = []
            for i in range(0, len(coords), 2):
                x = int(coords[i] * img_w)
                y = int(coords[i+1] * img_h)
                pts.append([x, y])

            polygons.append((cls, np.array(pts, dtype=np.int32)))

    return polygons


# ----------------------------
# Polygon → mask
# ----------------------------
def polygons_to_mask(polygons, shape):
    mask = np.zeros(shape[:2], dtype=np.uint8)

    for _, pts in polygons:
        cv2.fillPoly(mask, [pts], 1)

    return mask


# ----------------------------
# Boundary extraction
# ----------------------------
def extract_boundary(mask, thickness=2):
    kernel = np.ones((3, 3), np.uint8)
    boundary = cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, kernel)
    boundary = cv2.dilate(boundary, kernel, iterations=thickness)
    return boundary


# ----------------------------
# Check if image contains class
# ----------------------------
def contains_class(polygons, class_id):
    return any(cls == class_id for cls, _ in polygons)


# ----------------------------
# Get boundary (biased toward glass)
# ----------------------------
def get_random_boundary(image, polygons):
    if len(polygons) == 0:
        return None, None

    glass_polys = [p for p in polygons if p[0] == GLASS_CLASS_ID]

    # 70% chance to pick glass if available
    if glass_polys and random.random() < 0.7:
        _, pts = random.choice(glass_polys)
    else:
        _, pts = random.choice(polygons)

    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 1)

    boundary = extract_boundary(mask)
    boundary_pixels = image * boundary[:, :, None]

    return boundary, boundary_pixels


# ----------------------------
# Overlap check
# ----------------------------
def overlaps(a, b):
    return np.any((a + b) > 1)


# ----------------------------
# FakeMix core (with blending)
# ----------------------------
def fakemix(img1, polys1, img2, polys2, max_tries=20):
    h, w = img1.shape[:2]

    mask1 = polygons_to_mask(polys1, img1.shape)

    boundary, boundary_pixels = get_random_boundary(img2, polys2)

    if boundary is None:
        return img1

    for _ in range(max_tries):
        dx = random.randint(-w // 4, w // 4)
        dy = random.randint(-h // 4, h // 4)

        M = np.float32([[1, 0, dx], [0, 1, dy]])

        trans_boundary = cv2.warpAffine(boundary, M, (w, h))
        trans_pixels = cv2.warpAffine(boundary_pixels, M, (w, h))

        if overlaps(trans_boundary, mask1):
            continue

        # 🔥 soft blending (important)
        out = img1.copy()
        mask = trans_boundary.astype(bool)

        alpha = 0.6
        out[mask] = (
            alpha * img1[mask] +
            (1 - alpha) * trans_pixels[mask]
        ).astype(np.uint8)

        return out

    return img1


def has_any_class(polygons, target_classes):
    target_classes = set(target_classes)
    return any(cls in target_classes for cls, _ in polygons)

# ----------------------------
# Main augmentation pipeline
# ----------------------------
def augment_dataset(data_yaml_path, base_p=0.3, max_aug_per_image=1):
    train_img_dir, train_lbl_dir = resolve_paths(data_yaml_path)

    print("Train images:", train_img_dir)
    print("Train labels:", train_lbl_dir)

    image_paths = []
    for ext in ["*.jpg", "*.png", "*.jpeg"]:
        image_paths.extend(glob(os.path.join(train_img_dir, ext)))

    print(f"Found {len(image_paths)} images")
    total_aug = 0

    for img1_path in image_paths:
        txt1_path = img1_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"

        if not os.path.exists(txt1_path):
            continue

        img1 = cv2.imread(img1_path)
        if img1 is None:
            continue

        h, w = img1.shape[:2]
        polys1 = load_yolo_polygons(txt1_path, w, h)

        if has_any_class(polys1, SKIP_CLASSES):
                continue

        # 🔥 dynamic probability instead of hard filtering
        if contains_class(polys1, GLASS_CLASS_ID):
            p = base_p * 1.5   # more likely if glass present
        else:
            p = base_p * 0.3   # still possible but rare

        if random.random() > p:
            continue

        for i in range(max_aug_per_image):
            img2_path = random.choice(image_paths)
            txt2_path = img2_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"

            if not os.path.exists(txt2_path):
                continue

            img2 = cv2.imread(img2_path)
            if img2 is None:
                continue

            polys2 = load_yolo_polygons(txt2_path, w, h)

            out = fakemix(img1, polys1, img2, polys2)

            # Save
            base = os.path.basename(img1_path).rsplit(".", 1)[0]
            new_name = f"{base}_fm_{i}.jpg"

            out_img_path = os.path.join(train_img_dir, new_name)
            out_lbl_path = os.path.join(train_lbl_dir, new_name.replace(".jpg", ".txt"))

            cv2.imwrite(out_img_path, out)

            with open(txt1_path, "r") as f:
                label_data = f.read()

            with open(out_lbl_path, "w") as f:
                f.write(label_data)

            total_aug += 1

    print(f"✅ Done! Generated {total_aug} augmented images.")


# ----------------------------
# Run
# ----------------------------
augment_dataset(
    "./merged_dataset/merged_dataset/data.yaml",
    base_p=0.7,
    max_aug_per_image=2
)

In [ ]:
import os
from glob import glob

def delete_fakemix(data_yaml_path):
    import yaml

    # --- resolve paths (same correct logic as before) ---
    with open(data_yaml_path, "r") as f:
        data = yaml.safe_load(f)

    yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))
    base_path = data.get("path", "")

    if base_path:
        base_path = os.path.join(yaml_dir, base_path)
    else:
        base_path = yaml_dir

    train_img_dir = os.path.normpath(os.path.join(base_path, data["train"]))
    train_lbl_dir = train_img_dir.replace("images", "labels")

    print("Cleaning images in:", train_img_dir)
    print("Cleaning labels in:", train_lbl_dir)
    

    # --- find fakemix files ---
    img_patterns = ["*_fm_*.jpg", "*_fm_*.png", "*_fm_*.jpeg"]

    deleted = 0

    for pattern in img_patterns:
        for img_path in glob(os.path.join(train_img_dir, pattern)):
            lbl_path = os.path.join(
                train_lbl_dir,
                os.path.basename(img_path).rsplit(".", 1)[0] + ".txt"
            )

            # delete image
            os.remove(img_path)

            # delete label if exists
            if os.path.exists(lbl_path):
                os.remove(lbl_path)

            deleted += 1

    print(f"🧹 Deleted {deleted} FakeMix samples.")

delete_fakemix("./merged_dataset/merged_dataset/data.yaml")

In [ ]:
import os
from glob import glob

def analyze_fakemix_usage(data_yaml_path, target_class):
    import yaml

    # --- resolve paths correctly ---
    with open(data_yaml_path, "r") as f:
        data = yaml.safe_load(f)

    yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))
    base_path = data.get("path", "")

    if base_path:
        base_path = os.path.join(yaml_dir, base_path)
    else:
        base_path = yaml_dir

    train_img_dir = os.path.normpath(os.path.join(base_path, data["train"]))
    train_lbl_dir = train_img_dir.replace("images", "labels")

    # --- find all FakeMix images ---
    fm_images = []
    for ext in ["*.jpg", "*.png", "*.jpeg"]:
        fm_images.extend(glob(os.path.join(train_img_dir, f"*_fm_*{ext[-4:]}")))

    total_fm = len(fm_images)
    contains_target = 0

    for img_path in fm_images:
        txt_path = os.path.join(
            train_lbl_dir,
            os.path.basename(img_path).rsplit(".", 1)[0] + ".txt"
        )

        if not os.path.exists(txt_path):
            continue

        with open(txt_path, "r") as f:
            for line in f:
                cls = int(float(line.strip().split()[0]))
                if cls == target_class:
                    contains_target += 1
                    break

    print(f"Total FakeMix images: {total_fm}")
    print(f"FakeMix images containing class {target_class}: {contains_target}")

    if total_fm > 0:
        print(f"Percentage: {(contains_target / total_fm) * 100:.2f}%")

In [ ]:
import os
from glob import glob

def analyze_fakemix_usage(data_yaml_path, target_class):
    import yaml

    # --- resolve paths correctly ---
    with open(data_yaml_path, "r") as f:
        data = yaml.safe_load(f)

    yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))
    base_path = data.get("path", "")

    if base_path:
        base_path = os.path.join(yaml_dir, base_path)
    else:
        base_path = yaml_dir

    train_img_dir = os.path.normpath(os.path.join(base_path, data["train"]))
    train_lbl_dir = train_img_dir.replace("images", "labels")

    # --- find all FakeMix images ---
    fm_images = []
    for ext in ["*.jpg", "*.png", "*.jpeg"]:
        fm_images.extend(glob(os.path.join(train_img_dir, f"*_fm_*{ext[-4:]}")))

    total_fm = len(fm_images)
    contains_target = 0

    for img_path in fm_images:
        txt_path = os.path.join(
            train_lbl_dir,
            os.path.basename(img_path).rsplit(".", 1)[0] + ".txt"
        )

        if not os.path.exists(txt_path):
            continue

        with open(txt_path, "r") as f:
            for line in f:
                cls = int(float(line.strip().split()[0]))
                if cls == target_class:
                    contains_target += 1
                    break

    print(f"Total FakeMix images: {total_fm}")
    print(f"FakeMix images containing class {target_class}: {contains_target}")

    if total_fm > 0:
        print(f"Percentage: {(contains_target / total_fm) * 100:.2f}%")

analyze_fakemix_usage(
    "./merged_dataset/merged_dataset/data.yaml",
    target_class=6   # cigarettes
)